In [1]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline, interp1d
import plotly.graph_objects as go
from scipy.signal import correlate, find_peaks, butter, filtfilt, welch
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from plotly.subplots import make_subplots
import plotly.subplots as sp
import math
from scipy.integrate import cumulative_trapezoid
import os
from ipywidgets import HTML
from IPython.display import display


In [2]:
# Sujeito e caminhos
sujeito = "13"  # ex.: "01", "02", "03", ...

file_path_sp = f"Input_raw/{sujeito}(sl).csv"
file_path_vicon = f"Input_raw/{sujeito}(vc).xlsx"

# Carregar dados
df_sp = pd.read_csv(file_path_sp)  # Aceleração smartphone
df_vc = pd.read_excel(file_path_vicon, skiprows=3)

# Parâmetros
fs = 60  # Hz
dt = 1 / fs

# Saída
file_output = "Input_ML/"
os.makedirs(file_output, exist_ok=True)

print(f"Sujeito {sujeito} | SP: {file_path_sp} | Vicon: {file_path_vicon}")


Sujeito 13 | SP: Input_raw/13(sl).csv | Vicon: Input_raw/13(vc).xlsx


In [3]:
# =====================================================
# 1. Aceleração do Esterno - Smartphone - Interpolação
# =====================================================
Time_sp = df_sp["accelerometerTimestamp_sinceReboot(s)"].values
Signal_sp = df_sp["accelerometerAccelerationY(G)"].values * -9.80665
Signal_sp -= 9.80665
Time_diff_sp = np.diff(Time_sp)
Time_diff_sp = np.insert(Time_diff_sp, 0, 0)
Time_cumsum_sp = np.cumsum(Time_diff_sp)
if not np.all(np.diff(Time_cumsum_sp) > 0):
    unique_indices_sp = np.unique(Time_cumsum_sp, return_index=True)[1]
    Time_cumsum_sp = Time_cumsum_sp[unique_indices_sp]
    Signal_sp = Signal_sp[unique_indices_sp]
    sorted_indices_sp = np.argsort(Time_cumsum_sp)
    Time_cumsum_sp = Time_cumsum_sp[sorted_indices_sp]
    Signal_sp = Signal_sp[sorted_indices_sp]
New_Time_sp = np.arange(Time_cumsum_sp.min(), Time_cumsum_sp.max(), dt)
cs_sp = CubicSpline(Time_cumsum_sp, Signal_sp)
Signal_interpolated_sp = cs_sp(New_Time_sp)

# ========================================
# 2. Deslocamento do Esterno - Vicon - Interpolação
# ========================================
df_vc.columns = ['Frame', 'Time', 'Esterno_X', 'Esterno_Y', 'Esterno_Z', 'C7_X', 'C7_Y', 'C7_Z']
df_vc['Time'] = pd.to_numeric(df_vc['Time'], errors='coerce')
df_vc['Esterno_Z'] = pd.to_numeric(df_vc['Esterno_Z'], errors='coerce')
df_vc.dropna(subset=['Time', 'Esterno_Z'], inplace=True)
Time_vc = df_vc['Time'].to_numpy()
Signal_vc = df_vc['Esterno_Z'].to_numpy()
Time_diff_vc = np.diff(Time_vc)
Time_diff_vc = np.insert(Time_diff_vc, 0, 0)
Time_cumsum_vc = np.cumsum(Time_diff_vc)
if not np.all(np.diff(Time_cumsum_vc) > 0):
    unique_indices_vc = np.unique(Time_cumsum_vc, return_index=True)[1]
    Time_cumsum_vc = Time_cumsum_vc[unique_indices_vc]
    Signal_vc = Signal_vc[unique_indices_vc]
    sorted_indices_vc = np.argsort(Time_cumsum_vc)
    Time_cumsum_vc = Time_cumsum_vc[sorted_indices_vc]
    Signal_vc = Signal_vc[sorted_indices_vc]
New_Time_vc = np.arange(Time_cumsum_vc.min(), Time_cumsum_vc.max(), dt)
cs_linear = interp1d(Time_cumsum_vc, Signal_vc, kind='linear')
Signal_interpolated_vc = cs_linear(New_Time_vc)

# ========================================
# Plotagem lado a lado
# ========================================
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    "Smartphone - Aceleração Z",
    "Vicon - Deslocamento Z"
])

fig.add_trace(go.Scatter(x=Time_cumsum_sp, y=Signal_sp, mode='lines', name='Bruto Smartphone', line=dict(color='cyan')), row=1, col=1)
fig.add_trace(go.Scatter(x=New_Time_sp, y=Signal_interpolated_sp, mode='lines', name='Interpolado Smartphone', line=dict(color='red')), row=1, col=1)

fig.add_trace(go.Scatter(x=Time_cumsum_vc, y=Signal_vc, mode='lines', name='Bruto Vicon', line=dict(color='cyan')), row=1, col=2)
fig.add_trace(go.Scatter(x=New_Time_vc, y=Signal_interpolated_vc, mode='lines', name='Interpolado Vicon', line=dict(color='red')), row=1, col=2)

fig.update_layout(
    title="Comparação dos Sinais - Esterno Z",
    width=1500,
    height=400,
    template="plotly_dark",
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    xaxis=dict(title='Tempo (s)', showgrid=True, gridcolor='gray'),
    yaxis=dict(title='Sinal', showgrid=True, gridcolor='gray')
)

fig.show()

In [4]:
# === PARÂMETROS DO FILTRO ===
fc = 2  # Frequência de corte (Hz) – ajuste conforme necessário
order = 4  # Ordem do filtro

# === FUNÇÃO DE FILTRAGEM BUTTERWORTH ===
def butter_lowpass_filter(data, cutoff, fs, order=2):
    nyq = 0.5 * fs  # Frequência de Nyquist
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

# === APLICA O FILTRO AO SINAL INTERPOLADO ===
Signal_filtered_sp = butter_lowpass_filter(Signal_interpolated_sp, fc, fs, order)

Signal_filtered_sp -= np.mean(Signal_filtered_sp)
Signal_interpolated_sp -= np.mean(Signal_interpolated_sp)


# === PLOTAGEM DO SINAL BRUTO vs FILTRADO ===
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=New_Time_sp,
    y=Signal_interpolated_sp,
    mode='lines',
    name='Interpolado Bruto',
    line=dict(color='magenta')
))

fig.add_trace(go.Scatter(
    x=New_Time_sp,
    y=Signal_filtered_sp,
    mode='lines',
    name='Filtrado (Butterworth)',
    line=dict(color='lime')
))

fig.update_layout(
    title="Filtro no Sinal Interpolado do Smartphone (Esterno - Z)",
    xaxis_title="Tempo (s)",
    yaxis_title="Aceleração (m/s²)",
    template="plotly_dark",
    width=1500,
    height=400,
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(x=0.01, y=0.99)
)

fig.show()


In [5]:
# =====================================================
# Vicon — derivação + preparação para corte
# Execute após filtro do smartphone (e sempre que trocar de sujeito)
# =====================================================

def _derivar_vicon(disp_mm_arr, t_arr):
    disp_mm_arr = np.asarray(disp_mm_arr, dtype=float)
    disp_cm = disp_mm_arr / 10.0
    disp_cm_norm = disp_cm - disp_cm[0]
    disp_m = disp_mm_arr / 1000.0
    vel = np.gradient(disp_m, dt)
    acc = np.gradient(vel, dt)
    acc_f = butter_lowpass_filter(acc, fc, fs, order)
    acc_f = acc_f - np.mean(acc_f)
    return disp_cm_norm, acc, acc_f


_pipeline_key = (str(file_path_sp), str(file_path_vicon))
_trocou_sujeito = globals().get("_pipeline_key") != _pipeline_key
if _trocou_sujeito:
    for _k in list(globals().keys()):
        if _k.startswith("_orig_"):
            del globals()[_k]
    print(f"[prep] Novo sujeito/arquivos: {_pipeline_key[0]} | {_pipeline_key[1]}")

globals()["_pipeline_key"] = _pipeline_key

# Snapshot dos sinais interpolados/filtrados (sempre atualiza ao reexecutar esta célula)
_orig_New_Time_sp = np.asarray(New_Time_sp, dtype=float).copy()
_orig_Signal_interpolated_sp = np.asarray(Signal_interpolated_sp, dtype=float).copy()
_orig_Signal_filtered_sp = np.asarray(Signal_filtered_sp, dtype=float).copy()
_orig_New_Time_vc = np.asarray(New_Time_vc, dtype=float).copy()
_orig_Signal_interpolated_vc = np.asarray(Signal_interpolated_vc, dtype=float).copy()
_orig_Time_cumsum_vc = np.asarray(Time_cumsum_vc, dtype=float).copy()
_orig_Signal_vc = np.asarray(Signal_vc, dtype=float).copy()

New_Time_sp = _orig_New_Time_sp.copy()
Signal_interpolated_sp = _orig_Signal_interpolated_sp.copy()
Signal_filtered_sp = _orig_Signal_filtered_sp.copy()
New_Time_vc = _orig_New_Time_vc.copy()
Signal_interpolated_vc = _orig_Signal_interpolated_vc.copy()
Time_cumsum_vc = _orig_Time_cumsum_vc.copy()
Signal_vc = _orig_Signal_vc.copy()

corte_manual_aplicado = False
t_corte_sp = (float(New_Time_sp[0]), float(New_Time_sp[-1]))
t_corte_vc = (float(New_Time_vc[0]), float(New_Time_vc[-1]))

disp_cm_norm, acc_vicon_m_s2, acc_vicon_filt_m_s2 = _derivar_vicon(
    Signal_interpolated_vc, New_Time_vc
)
t_vc = np.asarray(New_Time_vc, dtype=float)

print(
    f"[prep] Smartphone: {len(New_Time_sp)} amostras | "
    f"{New_Time_sp[0]:.2f} → {New_Time_sp[-1]:.2f} s"
)
print(
    f"[prep] Vicon: {len(t_vc)} amostras | "
    f"{t_vc[0]:.2f} → {t_vc[-1]:.2f} s | acel. derivada pronta para corte"
)

fig_prep = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Smartphone — acel. Y filtrada",
        "Vicon — acel. derivada filtrada (Esterno Z)",
    ),
)
fig_prep.add_trace(
    go.Scatter(
        x=New_Time_sp, y=Signal_filtered_sp, mode="lines",
        name="SP filtrado", line=dict(color="lime"),
    ),
    row=1, col=1,
)
fig_prep.add_trace(
    go.Scatter(
        x=t_vc, y=acc_vicon_filt_m_s2, mode="lines",
        name="Vicon filtrado", line=dict(color="orange"),
    ),
    row=1, col=2,
)
fig_prep.update_layout(
    title="Pré-corte — confira os sinais antes da célula de corte manual",
    template="plotly_dark",
    width=1500,
    height=420,
    plot_bgcolor="black",
    paper_bgcolor="black",
    font=dict(color="white"),
)
fig_prep.update_xaxes(title_text="Tempo (s)")
fig_prep.update_yaxes(title_text="Aceleração (m/s²)")
fig_prep.show()

print("Próximo passo: execute a célula de CORTE MANUAL (4 cliques).")



[prep] Smartphone: 2981 amostras | 0.00 → 49.67 s
[prep] Vicon: 2031 amostras | 0.00 → 33.83 s | acel. derivada pronta para corte


Próximo passo: execute a célula de CORTE MANUAL (4 cliques).


In [ ]:
# =====================================================
# Corte manual — 2 gráficos (smartphone + Vicon)
# Requer: células 1→3 + preparação Vicon acima
# =====================================================

if "_orig_New_Time_sp" not in globals():
    raise RuntimeError(
        "Execute a célula anterior (Vicon — derivação + preparação) antes desta."
    )

if "acc_vicon_filt_m_s2" not in globals():
    raise RuntimeError(
        "Derivação do Vicon ausente. Execute a célula de preparação antes desta."
    )


def _fig_layout_base(fig, titulo, t_min, t_max):
    fig.update_layout(
        title=titulo,
        template="plotly_dark",
        width=1500,
        height=380,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        legend=dict(x=0.01, y=0.99),
        dragmode="pan",
        uirevision="corte_manual",
    )
    fig.update_xaxes(
        title_text="Tempo (s)",
        range=[float(t_min), float(t_max)],
        autorange=False,
        fixedrange=False,
    )
    fig.update_yaxes(autorange=True)
    return fig


def _shapes_painel(marcadores):
    return [
        dict(
            type="line", x0=t, x1=t, y0=0, y1=1, yref="paper", xref="x",
            line=dict(color=cor, width=2, dash="dash"),
        )
        for t, cor in marcadores
    ]


def _aplicar_shapes_sem_zoom(fig_w, marcadores):
    xr = fig_w.layout.xaxis.range
    yr = fig_w.layout.yaxis.range
    fig_w.layout.shapes = _shapes_painel(marcadores)
    if xr is not None:
        fig_w.layout.xaxis.range = list(xr)
    if yr is not None:
        fig_w.layout.yaxis.range = list(yr)


def _aplicar_corte_manual():
    global New_Time_sp, Signal_interpolated_sp, Signal_filtered_sp
    global New_Time_vc, Signal_interpolated_vc
    global Time_cumsum_vc, Signal_vc
    global acc_vicon_m_s2, acc_vicon_filt_m_s2
    global corte_manual_aplicado, t_corte_sp, t_corte_vc

    t_sp0, t_sp1 = sorted(_cliques_corte["sp"])
    t_vc0, t_vc1 = sorted(_cliques_corte["vc"])
    t_corte_sp = (float(t_sp0), float(t_sp1))
    t_corte_vc = (float(t_vc0), float(t_vc1))

    m_sp = (New_Time_sp >= t_sp0) & (New_Time_sp <= t_sp1)
    m_vc = (New_Time_vc >= t_vc0) & (New_Time_vc <= t_vc1)
    m_vc_raw = (Time_cumsum_vc >= t_vc0) & (Time_cumsum_vc <= t_vc1)

    New_Time_sp = np.asarray(New_Time_sp, dtype=float)[m_sp]
    Signal_interpolated_sp = np.asarray(Signal_interpolated_sp, dtype=float)[m_sp]
    Signal_filtered_sp = np.asarray(Signal_filtered_sp, dtype=float)[m_sp]

    New_Time_vc = np.asarray(New_Time_vc, dtype=float)[m_vc]
    Signal_interpolated_vc = np.asarray(Signal_interpolated_vc, dtype=float)[m_vc]
    Time_cumsum_vc = np.asarray(Time_cumsum_vc, dtype=float)[m_vc_raw]
    Signal_vc = np.asarray(Signal_vc, dtype=float)[m_vc_raw]

    disp_n, acc_r, acc_f = _derivar_vicon(Signal_interpolated_vc, New_Time_vc)
    acc_vicon_m_s2 = acc_r
    acc_vicon_filt_m_s2 = acc_f
    corte_manual_aplicado = True

    print(f"Corte smartphone: {t_sp0:.3f} s → {t_sp1:.3f} s ({m_sp.sum()} amostras)")
    print(f"Corte Vicon:      {t_vc0:.3f} s → {t_vc1:.3f} s ({m_vc.sum()} amostras)")


def _plot_resumo_pos_corte():
    disp_n, acc_r, acc_f = _derivar_vicon(Signal_interpolated_vc, New_Time_vc)
    t_vc = np.asarray(New_Time_vc, dtype=float)

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.08,
        subplot_titles=(
            "Smartphone — acel. Y (após corte)",
            "Vicon — acel. derivada filtrada (após corte)",
        ),
    )
    fig.add_trace(go.Scatter(
        x=New_Time_sp, y=Signal_filtered_sp, mode="lines",
        name="Smartphone filtrado", line=dict(color="lime"),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=t_vc, y=acc_f, mode="lines",
        name="Vicon filtrado", line=dict(color="orange"),
    ), row=2, col=1)
    fig.update_layout(
        title="Resumo após corte manual",
        template="plotly_dark", width=1500, height=650,
        plot_bgcolor="black", paper_bgcolor="black", font=dict(color="white"),
    )
    fig.update_xaxes(title_text="Tempo (s)", row=2, col=1)
    fig.update_yaxes(title_text="Aceleração (m/s²)", row=1, col=1)
    fig.update_yaxes(title_text="Aceleração (m/s²)", row=2, col=1)
    fig.show()


# Restaura sinais completos (reexecutar esta célula = novo corte, sem cortar de novo o anterior)
New_Time_sp = _orig_New_Time_sp.copy()
Signal_interpolated_sp = _orig_Signal_interpolated_sp.copy()
Signal_filtered_sp = _orig_Signal_filtered_sp.copy()
New_Time_vc = _orig_New_Time_vc.copy()
Signal_interpolated_vc = _orig_Signal_interpolated_vc.copy()
Time_cumsum_vc = _orig_Time_cumsum_vc.copy()
Signal_vc = _orig_Signal_vc.copy()

corte_manual_aplicado = False
t_corte_sp = (float(New_Time_sp[0]), float(New_Time_sp[-1]))
t_corte_vc = (float(New_Time_vc[0]), float(New_Time_vc[-1]))

disp_cm_norm, acc_vicon_m_s2, acc_vicon_filt_m_s2 = _derivar_vicon(
    Signal_interpolated_vc, New_Time_vc
)
t_vc = np.asarray(New_Time_vc, dtype=float)

print(f"Vicon | amostras: {len(t_vc)} | dt = {dt:.6f} s | fs = {fs} Hz")

_cliques_corte = {"sp": [], "vc": []}
_marcadores = {"sp": [], "vc": []}

_fig_sp = go.Figure()
_fig_sp.add_trace(go.Scatter(
    x=New_Time_sp, y=Signal_filtered_sp, mode="lines",
    name="Acel. Y filtrada", line=dict(color="lime"),
))
_fig_layout_base(
    _fig_sp,
    "CORTE SMARTPHONE — clique 2×: início e fim (verde / vermelho)",
    New_Time_sp[0], New_Time_sp[-1],
)
fig_corte_sp = go.FigureWidget(_fig_sp)

_fig_vc = go.Figure()
_fig_vc.add_trace(go.Scatter(
    x=t_vc, y=acc_vicon_filt_m_s2, mode="lines",
    name="Acel. derivada filtrada", line=dict(color="orange"),
))
_fig_layout_base(
    _fig_vc,
    "CORTE VICON — clique 2×: início e fim (verde / vermelho)",
    t_vc[0], t_vc[-1],
)
fig_corte_vc = go.FigureWidget(_fig_vc)

print(
    "①② Gráfico SMARTPHONE: clique início e fim\n"
    "③④ Gráfico VICON: clique início e fim\n"
    "(Após os 4 cliques, abre o resumo com os dados cortados.)"
)
display(HTML("<b>Dois gráficos separados</b> — use pan/zoom se precisar; os cliques não alteram o zoom."))


def _tentar_finalizar():
    if len(_cliques_corte["sp"]) == 2 and len(_cliques_corte["vc"]) == 2:
        _aplicar_corte_manual()
        _plot_resumo_pos_corte()
        print("Corte concluído. Variáveis globais atualizadas para as próximas células.")


def _handler_clique(painel, fig_w):
    def _cb(trace, points, selector):
        if not points.xs:
            return
        t = float(points.xs[0])
        lst = _cliques_corte[painel]
        if len(lst) >= 2:
            print(f"[{painel}] Já tem início e fim. Reexecute a célula para novo corte.")
            return
        lst.append(t)
        cor = "lime" if len(lst) == 1 else "red"
        _marcadores[painel].append((t, cor))
        print(f"[{painel}] t = {t:.3f} s ({'início' if len(lst) == 1 else 'fim'})")
        _aplicar_shapes_sem_zoom(fig_w, _marcadores[painel])
        fig_w.data[0].on_click(_cb)
        _tentar_finalizar()
    return _cb


_cb_sp = _handler_clique("sp", fig_corte_sp)
_cb_vc = _handler_clique("vc", fig_corte_vc)
fig_corte_sp.data[0].on_click(_cb_sp)
fig_corte_vc.data[0].on_click(_cb_vc)

display(fig_corte_sp)
display(fig_corte_vc)



Vicon | amostras: 2031 | dt = 0.016667 s | fs = 60 Hz
①② Gráfico SMARTPHONE: clique início e fim
③④ Gráfico VICON: clique início e fim
(Após os 4 cliques, abre o resumo com os dados cortados.)


HTML(value='<b>Dois gráficos separados</b> — use pan/zoom se precisar; os cliques não alteram o zoom.')

FigureWidget({
    'data': [{'line': {'color': 'lime'},
              'mode': 'lines',
              'name': 'Acel. Y filtrada',
              'type': 'scatter',
              'uid': 'a89967a2-0742-441e-9f98-a2d3e0d174ea',
              'x': array([0.00000000e+00, 1.66666667e-02, 3.33333333e-02, ..., 4.96333333e+01,
                          4.96500000e+01, 4.96666667e+01]),
              'y': array([-3.60359072, -3.59901141, -3.59424186, ..., -5.28627076, -5.2807739 ,
                          -5.27576683])}],
    'layout': {'dragmode': 'pan',
               'font': {'color': 'white'},
               'height': 380,
               'legend': {'x': 0.01, 'y': 0.99},
               'paper_bgcolor': 'black',
               'plot_bgcolor': 'black',
               'template': '...',
               'title': {'text': 'CORTE SMARTPHONE — clique 2×: início e fim (verde / vermelho)'},
               'uirevision': 'corte_manual',
               'width': 1500,
               'xaxis': {'autorange': 

FigureWidget({
    'data': [{'line': {'color': 'orange'},
              'mode': 'lines',
              'name': 'Acel. derivada filtrada',
              'type': 'scatter',
              'uid': '4ecc2a76-05fa-482a-b5a0-bafa6054b76c',
              'x': array([0.00000000e+00, 1.66666667e-02, 3.33333333e-02, ..., 3.38000000e+01,
                          3.38166667e+01, 3.38333333e+01]),
              'y': array([-0.03448586, -0.02808404, -0.02231503, ..., -0.34850472, -0.36293017,
                          -0.37326174])}],
    'layout': {'dragmode': 'pan',
               'font': {'color': 'white'},
               'height': 380,
               'legend': {'x': 0.01, 'y': 0.99},
               'paper_bgcolor': 'black',
               'plot_bgcolor': 'black',
               'template': '...',
               'title': {'text': 'CORTE VICON — clique 2×: início e fim (verde / vermelho)'},
               'uirevision': 'corte_manual',
               'width': 1500,
               'xaxis': {'autorang

In [8]:
# =====================================================
# Alinhamento — cross-correlation nos dados CORTADOS
# Tempo absoluto SP e Vicon são diferentes → usa tempo RELATIVO ao corte
# =====================================================

if not corte_manual_aplicado:
    raise RuntimeError(
        "Execute a célula de corte manual (4 cliques) antes desta célula."
    )

t_sp_abs = np.asarray(New_Time_sp, dtype=float)
y_sp_full = np.asarray(Signal_filtered_sp, dtype=float)
t_vc_abs = np.asarray(New_Time_vc, dtype=float)
y_vc_full = np.asarray(acc_vicon_filt_m_s2, dtype=float)

print(f"[corte] Smartphone: {t_corte_sp[0]:.3f} → {t_corte_sp[1]:.3f} s ({len(t_sp_abs)} amostras)")
print(f"[corte] Vicon:      {t_corte_vc[0]:.3f} → {t_corte_vc[1]:.3f} s ({len(t_vc_abs)} amostras)")

# Tempo RELATIVO a cada corte (t=0 no início do recorte) — bases temporais diferentes!
t_sp_rel = t_sp_abs - t_sp_abs[0]
t_vc_rel = t_vc_abs - t_vc_abs[0]

if len(t_sp_abs) != len(y_sp_full) or len(t_vc_abs) != len(y_vc_full):
    raise ValueError(
        f"Tamanhos inconsistentes — reexecute a célula de corte (4 cliques): "
        f"SP tempo={len(t_sp_abs)} sinal={len(y_sp_full)}, "
        f"Vicon tempo={len(t_vc_abs)} sinal={len(y_vc_full)}."
    )

dur_sp = float(t_sp_rel[-1])
dur_vc = float(t_vc_rel[-1])

# Mesma grade 60 Hz após o corte → trecho comum = primeiros N amostras (evita NaN do arange)
n_comum = int(min(len(y_sp_full), len(y_vc_full)))
dur_comum = float((n_comum - 1) * dt) if n_comum > 1 else 0.0

if dur_comum < 20.0:
    raise ValueError(
        f"Cortes muito curtos: SP={dur_sp:.2f}s, Vicon={dur_vc:.2f}s, comum={dur_comum:.2f}s. "
        "Refaça os cliques para ter pelo menos 20 s em cada um."
    )

t_comum = np.arange(n_comum, dtype=float) * dt
sp_c = np.asarray(y_sp_full[:n_comum], dtype=float)
vc_c = np.asarray(y_vc_full[:n_comum], dtype=float)

if np.any(~np.isfinite(sp_c)) or np.any(~np.isfinite(vc_c)):
    raise ValueError(
        "Sinal cortado contém NaN/Inf. Reexecute a célula de corte (4 cliques) e depois esta."
    )

print(f"[corte] Cross-corr no trecho comum: {len(t_comum)} amostras = {dur_comum:.2f} s")

# Sem inverter sinal — mesmo sinal do gráfico de corte
inverter_sp = False
sp_uso = sp_c.copy()

r0, _ = pearsonr(sp_uso, vc_c)
print(f"[info] Pearson bruto (sem inverter): {r0:.3f}")

sp_n = sp_uso - np.mean(sp_uso)
vc_n = vc_c - np.mean(vc_c)

# Cross-correlation (convenção Equação_utt)
corr = correlate(sp_n, vc_n, mode="full")
lag_idx = int(np.argmax(corr) - (len(sp_n) - 1))
max_lag = int(min(15 * fs, len(sp_n) // 2))
lag_idx = int(np.clip(lag_idx, -max_lag, max_lag))


def _aplicar_lag_equacao(sp, vc, t, lag):
    """Mesma lógica do Equação_utt.ipynb após correlate."""
    sp = np.asarray(sp, dtype=float)
    vc = np.asarray(vc, dtype=float)
    t = np.asarray(t, dtype=float)
    if lag > 0:
        sp_a = sp[lag:]
        vc_a = vc[: len(sp_a)]
        t_a = t[: len(sp_a)]
    elif lag < 0:
        vc_a = vc[-lag:]
        sp_a = sp[: len(vc_a)]
        t_a = t[: len(vc_a)]
    else:
        sp_a, vc_a, t_a = sp, vc, t
    return t_a, sp_a, vc_a


# Testa lag e -lag; fica o melhor Pearson
melhor = None
for lag_try in (lag_idx, -lag_idx):
    t_a, sp_a, vc_a = _aplicar_lag_equacao(sp_uso, vc_c, t_comum, lag_try)
    if len(sp_a) < int(0.5 * fs):
        continue
    r_try, _ = pearsonr(sp_a, vc_a)
    if melhor is None or r_try > melhor[0]:
        melhor = (r_try, lag_try, t_a, sp_a, vc_a)

if melhor is None:
    raise ValueError("Alinhamento falhou nos dados cortados.")

r_val, lag_idx, t_al, sp_alinhado, y_vc_plot = melhor
align_shift_s = float(lag_idx) * dt  # para exportação no eixo do Vicon

t_plot = t_al - t_al[0]
print(f"Lag aplicado: {lag_idx} quadros ({align_shift_s:.3f} s)")
print(f"Correlação de Pearson (após alinhar): {r_val:.3f}")

fig_alinhado = go.Figure()
fig_alinhado.add_trace(go.Scatter(
    x=t_plot, y=y_vc_plot, mode="lines", name="Vicon (cortado)",
    line=dict(color="orange", width=2),
))
fig_alinhado.add_trace(go.Scatter(
    x=t_plot, y=sp_alinhado, mode="lines", name="Smartphone (cortado + alinhado)",
    line=dict(color="lime", width=2),
))
fig_alinhado.update_layout(
    title=f"Acelerações alinhadas — trecho cortado em comum (lag = {lag_idx} quadros)",
    xaxis_title="Tempo (s, t=0)",
    yaxis_title="Aceleração (m/s²)",
    template="plotly_dark",
    width=1500,
    height=450,
    plot_bgcolor="black",
    paper_bgcolor="black",
    font=dict(color="white"),
    legend=dict(x=0.01, y=0.99),
)
fig_alinhado.show()


[corte] Smartphone: 7.817 → 43.167 s (2122 amostras)
[corte] Vicon:      0.567 → 33.700 s (1989 amostras)
[corte] Cross-corr no trecho comum: 1989 amostras = 33.13 s
[info] Pearson bruto (sem inverter): -0.550
Lag aplicado: 90 quadros (1.500 s)
Correlação de Pearson (após alinhar): 0.994


In [ ]:
# =====================================================
# Exportar CSV — dados alinhados para Deep Learning
# Time (s, t=0) + 6 smartphone (bruto) + 1 Vicon deslocamento (bruto)
# =====================================================

def _prep_cumsum(time_arr, *signals):
    """Tempo cumulativo + sinais com mesma lógica de deduplicação da célula 2."""
    t = np.asarray(time_arr, dtype=float)
    sigs = [np.asarray(s, dtype=float) for s in signals]
    td = np.insert(np.diff(t), 0, 0.0)
    tc = np.cumsum(td)
    if not np.all(np.diff(tc) > 0):
        u = np.unique(tc, return_index=True)[1]
        tc = tc[u]
        sigs = [s[u] for s in sigs]
        order = np.argsort(tc)
        tc = tc[order]
        sigs = [s[order] for s in sigs]
    return (tc,) + tuple(sigs)


def _interp_bruto_sp(time_cum, signal, t_query):
    f = interp1d(
        time_cum, np.asarray(signal, dtype=float),
        kind="linear", bounds_error=False, fill_value=np.nan,
    )
    return f(np.asarray(t_query, dtype=float))


def _deslocar_sp_no_grid(y, lag, n_out):
    out = np.full(n_out, np.nan, dtype=float)
    y = np.asarray(y, dtype=float)
    if lag > 0:
        n_put = n_out - lag
        if n_put > 0:
            out[:n_put] = y[lag:lag + n_put]
    elif lag < 0:
        s = -lag
        n_put = n_out - s
        if n_put > 0:
            out[s:s + n_put] = y[:n_put]
    else:
        n_put = min(n_out, len(y))
        out[:n_put] = y[:n_put]
    return out


if not corte_manual_aplicado:
    raise RuntimeError("Execute o corte manual na célula anterior antes de exportar.")

# Smartphone bruto (apenas dentro do corte; tempo absoluto do smartphone)
_t_acc, _accX, _accY, _accZ = _prep_cumsum(
    df_sp["accelerometerTimestamp_sinceReboot(s)"].values,
    df_sp["accelerometerAccelerationX(G)"].values * -9.80665,
    df_sp["accelerometerAccelerationY(G)"].values * -9.80665 - 9.80665,
    df_sp["accelerometerAccelerationZ(G)"].values * -9.80665,
)
_m_acc = (_t_acc >= t_corte_sp[0]) & (_t_acc <= t_corte_sp[1])
_t_acc, _accX, _accY, _accZ = _t_acc[_m_acc], _accX[_m_acc], _accY[_m_acc], _accZ[_m_acc]

_t_gyro, _gyroX, _gyroY, _gyroZ = _prep_cumsum(
    df_sp["gyroTimestamp_sinceReboot(s)"].values,
    df_sp["gyroRotationX(rad/s)"].values,
    df_sp["gyroRotationY(rad/s)"].values,
    df_sp["gyroRotationZ(rad/s)"].values,
)
_m_gyro = (_t_gyro >= t_corte_sp[0]) & (_t_gyro <= t_corte_sp[1])
_t_gyro, _gyroX, _gyroY, _gyroZ = (
    _t_gyro[_m_gyro], _gyroX[_m_gyro], _gyroY[_m_gyro], _gyroZ[_m_gyro],
)

# Tempo RELATIVO de cada sinal (t=0 no início de cada corte)
# Smartphone bruto → tempo relativo ao início do corte SP
_t_acc_rel = _t_acc - float(t_corte_sp[0])
_t_gyro_rel = _t_gyro - float(t_corte_sp[0])

# Grid comum (mesmo usado no alinhamento)
N = len(t_comum)

sp_bruto = {
    "accX_m_s2": _interp_bruto_sp(_t_acc_rel, _accX, t_comum),
    "accY_m_s2": _interp_bruto_sp(_t_acc_rel, _accY, t_comum),
    "accZ_m_s2": _interp_bruto_sp(_t_acc_rel, _accZ, t_comum),
    "gyroX_rad_s": _interp_bruto_sp(_t_gyro_rel, _gyroX, t_comum),
    "gyroY_rad_s": _interp_bruto_sp(_t_gyro_rel, _gyroY, t_comum),
    "gyroZ_rad_s": _interp_bruto_sp(_t_gyro_rel, _gyroZ, t_comum),
}

for _k in sp_bruto:
    sp_bruto[_k] = _deslocar_sp_no_grid(sp_bruto[_k], lag_idx, N)

# Vicon: deslocamento cortado (mm → cm) no MESMO grid (tempo relativo do Vicon)
_t_vc_rel_full = np.asarray(New_Time_vc, dtype=float) - float(t_corte_vc[0])
vicon_bruto_cm = _interp_bruto_sp(
    _t_vc_rel_full, np.asarray(Signal_interpolated_vc, dtype=float) / 10.0, t_comum,
)

# Cortar na interseção válida
mask_ok = np.isfinite(vicon_bruto_cm)
for _v in sp_bruto.values():
    mask_ok &= np.isfinite(_v)

if not np.any(mask_ok):
    raise ValueError("Nenhum instante com smartphone e Vicon válidos após o alinhamento.")

idx_valid = np.where(mask_ok)[0]
i0, i1 = int(idx_valid[0]), int(idx_valid[-1])

time_s = (t_comum[i0:i1 + 1] - t_comum[i0]).astype(float)

vicon_export_cm = vicon_bruto_cm[i0:i1 + 1].astype(float)
vicon_export_cm = vicon_export_cm - vicon_export_cm[0]

df_export = pd.DataFrame({
    "Time": time_s,
    "accX_m_s2": sp_bruto["accX_m_s2"][i0:i1 + 1],
    "accY_m_s2": sp_bruto["accY_m_s2"][i0:i1 + 1],
    "accZ_m_s2": sp_bruto["accZ_m_s2"][i0:i1 + 1],
    "gyroX_rad_s": sp_bruto["gyroX_rad_s"][i0:i1 + 1],
    "gyroY_rad_s": sp_bruto["gyroY_rad_s"][i0:i1 + 1],
    "gyroZ_rad_s": sp_bruto["gyroZ_rad_s"][i0:i1 + 1],
    "vicon_esternoZ_cm": vicon_export_cm,
})

# Identificador do sujeito a partir do arquivo (ex.: "01(sl).csv" → "01")
_sujeito = os.path.basename(file_path_sp).split("(")[0]
os.makedirs(file_output, exist_ok=True)
_csv_path = os.path.join(file_output, f"{_sujeito}_alinhado_ml.csv")
df_export.to_csv(_csv_path, index=False)

print(f"CSV exportado: {_csv_path}")
print(f"Amostras: {len(df_export)} | duração: {time_s[-1]:.3f} s | lag: {lag_idx} quadros")
print(f"Colunas: {list(df_export.columns)}")
df_export.head()

CSV exportado: Input_ML/29_alinhado_ml.csv
Amostras: 2028 | duração: 33.783 s | lag: -2 quadros
Colunas: ['Time', 'accX_m_s2', 'accY_m_s2', 'accZ_m_s2', 'gyroX_rad_s', 'gyroY_rad_s', 'gyroZ_rad_s', 'vicon_esternoZ_cm']


,Time,accX_m_s2,accY_m_s2,accZ_m_s2,gyroX_rad_s,gyroY_rad_s,gyroZ_rad_s,vicon_esternoZ_cm
0,0.000000,-0.293993,-0.620333,-3.443665,0.004025,0.023372,0.010075,0.000000
1,0.016667,-0.265882,-0.631073,-3.453096,0.003106,0.041009,0.023011,0.003053
2,0.033333,-0.200059,-0.606893,-3.467403,0.001210,0.047517,0.027367,0.003907
3,0.050000,-0.197706,-0.607830,-3.507355,0.000510,0.041572,0.028465,0.005663
4,0.066667,-0.137815,-0.616738,-3.477199,0.002187,0.041109,0.025490,0.003577
